# DA4 Assignment 2 — Regression Analysis
**Question:** To what extent does economic activity cause CO2 emissions?

We estimate 6 models using ln(CO2 pc) as the dependent variable and ln(GDP pc) as the
main regressor, then add urbanization as a confounder to models 1, 4, and 6.

In [ ]:
import pandas as pd
import numpy as np
import pyfixest as pf
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('../data/wdi_clean.csv')
df = df.rename(columns={'Country Code': 'country', 'Country Name': 'country_name'})
print(f'{df.country.nunique()} countries, {df.year.min()}-{df.year.max()}')

## 1. Cross-section OLS

In [ ]:
# Find last year with good coverage
last_yr = df.dropna(subset=['ln_gdp_pc', 'ln_co2_pc']).year.max()
print(f'Last year with data: {last_yr}')

cs_2005 = df.query('year == 2005').dropna(subset=['ln_gdp_pc', 'ln_co2_pc'])
cs_last = df.query('year == @last_yr').dropna(subset=['ln_gdp_pc', 'ln_co2_pc'])

print(f'Cross-section 2005: N = {len(cs_2005)}')
print(f'Cross-section {last_yr}: N = {len(cs_last)}')

In [ ]:
# Model 1: OLS 2005
m1 = pf.feols('ln_co2_pc ~ ln_gdp_pc', data=cs_2005, vcov='HC1')

# Model 2: OLS last year
m2 = pf.feols('ln_co2_pc ~ ln_gdp_pc', data=cs_last, vcov='HC1')

pf.etable([m1, m2], model_heads=['OLS 2005', f'OLS {last_yr}'], head_order='h')

## 2. First Difference models

We take first differences to remove time-invariant country characteristics.
All FD models include year dummies as an aggregate time trend.

In [ ]:
# Sort and compute first differences
df = df.sort_values(['country', 'year'])
g = df.groupby('country')

df['d_ln_co2_pc'] = g['ln_co2_pc'].diff()
df['d_ln_gdp_pc'] = g['ln_gdp_pc'].diff()
df['d_urban_pct'] = g['urban_pct'].diff()

# Lags of d_ln_gdp_pc (for FD with lags)
for i in range(1, 7):
    df[f'd_ln_gdp_pc_L{i}'] = g['d_ln_gdp_pc'].shift(i)

# Also need lags of d_urban_pct for confounder model 4
for i in range(1, 3):
    df[f'd_urban_pct_L{i}'] = g['d_urban_pct'].shift(i)

print(f'First differences computed. Non-null d_ln_co2_pc: {df.d_ln_co2_pc.notna().sum()}')

In [ ]:
# Model 3: FD, time trend, no lags
m3 = pf.feols('d_ln_co2_pc ~ d_ln_gdp_pc + C(year)',
              data=df, vcov={'CRV1': 'country'})

# Model 4: FD, time trend, 2-year lags
m4 = pf.feols('d_ln_co2_pc ~ d_ln_gdp_pc + d_ln_gdp_pc_L1 + d_ln_gdp_pc_L2 + C(year)',
              data=df, vcov={'CRV1': 'country'})

# Model 5: FD, time trend, 6-year lags
lag_vars = ' + '.join([f'd_ln_gdp_pc_L{i}' for i in range(1, 7)])
m5 = pf.feols(f'd_ln_co2_pc ~ d_ln_gdp_pc + {lag_vars} + C(year)',
              data=df, vcov={'CRV1': 'country'})

pf.etable([m3, m4, m5],
          model_heads=['FD no lags', 'FD 2 lags', 'FD 6 lags'],
          head_order='h',
          drop='year')

### Cumulative (long-run) effects for FD models

The individual lag coefficients show how GDP changes propagate over time.
The cumulative effect (sum of contemporaneous + all lags) gives the **total long-run elasticity**.

In [ ]:
# Cumulative effects
for name, model, n_lags in [('M3 (no lags)', m3, 0), ('M4 (2 lags)', m4, 2), ('M5 (6 lags)', m5, 6)]:
    coefs = model.coef()
    gdp_vars = ['d_ln_gdp_pc'] + [f'd_ln_gdp_pc_L{i}' for i in range(1, n_lags + 1)]
    cumul = sum(coefs[v] for v in gdp_vars if v in coefs.index)
    print(f'{name}: cumulative effect = {cumul:.4f}')

## 3. Fixed Effects model

In [ ]:
# Model 6: FE with country and year fixed effects
m6 = pf.feols('ln_co2_pc ~ ln_gdp_pc + C(year) | country',
              data=df, vcov={'CRV1': 'country'})

pf.etable([m6], model_heads=['FE'], head_order='h', drop='year')

## 4. Summary table: all 6 models

In [ ]:
pf.etable([m1, m2, m3, m4, m5, m6],
          model_heads=['OLS 2005', f'OLS {last_yr}', 'FD', 'FD 2 lags', 'FD 6 lags', 'FE'],
          head_order='h',
          drop='year|C\(year\)',
          show_se_type=False)

## 5. Adding the confounder: Urbanization

Urbanization (% urban population) is a potential confounder: it drives both GDP growth
(through industrialization) and CO2 emissions (through energy-intensive urban infrastructure).
We add it to models 1 (OLS 2005), 4 (FD 2 lags), and 6 (FE).

In [ ]:
# Model 1c: OLS 2005 + urbanization
m1c = pf.feols('ln_co2_pc ~ ln_gdp_pc + urban_pct', data=cs_2005, vcov='HC1')

# Model 4c: FD 2 lags + urbanization (also differenced, with lags)
m4c = pf.feols('d_ln_co2_pc ~ d_ln_gdp_pc + d_ln_gdp_pc_L1 + d_ln_gdp_pc_L2 '
               '+ d_urban_pct + d_urban_pct_L1 + d_urban_pct_L2 + C(year)',
               data=df, vcov={'CRV1': 'country'})

# Model 6c: FE + urbanization
m6c = pf.feols('ln_co2_pc ~ ln_gdp_pc + urban_pct + C(year) | country',
               data=df, vcov={'CRV1': 'country'})

pf.etable([m1, m1c, m4, m4c, m6, m6c],
          model_heads=['OLS 2005', 'OLS 2005 + conf.', 'FD 2 lags', 'FD 2 lags + conf.',
                       'FE', 'FE + conf.'],
          head_order='h',
          drop='year|C\\(year\\)',
          show_se_type=False)